# Real-ESRGAN Video Upscaler for Kaggle GPU

A clean Kaggle notebook for **1080p → 4K (2×)** video upscaling.

### Features
- Uses `RealESRGAN_x2plus`, appropriate for 2× upscaling.
- Uses FP16 by default for better T4 performance.
- Uses **both T4 GPUs in parallel** when two GPUs are available.
- Automatically detects the source FPS.
- Preserves the original audio when possible.
- Keeps temporary files in `/kaggle/working`.
- Produces `/kaggle/working/upscaled_video.mp4`.

**For your current video:** `/kaggle/input/datasets/dhruvmatliwala/my-video/scene5.mp4`


## 1. Settings

Upload/attach your video to the Kaggle notebook first. The path below is already set for your `scene5.mp4`.


In [ ]:
# ===== USER SETTINGS =====

INPUT_VIDEO = "/kaggle/input/datasets/dhruvmatliwala/my-video/scene5.mp4"

# Your source is 1920x1080, so 2x produces 3840x2160 (4K).
SCALE = 2

# Native 2x Real-ESRGAN model.
MODEL = "RealESRGAN_x2plus"

# Final H.264 quality.
# Lower CRF = higher quality / larger file.
CRF = 18
PRESET = "medium"

OUTPUT_VIDEO = "/kaggle/working/upscaled_video.mp4"
WORK_DIR = "/kaggle/working/realesrgan_video"

print("Input :", INPUT_VIDEO)
print("Scale :", SCALE)
print("Model :", MODEL)
print("Output:", OUTPUT_VIDEO)


## 2. Check the Kaggle GPUs


In [ ]:
import os
import subprocess
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No NVIDIA GPU detected. In Kaggle select Settings -> Accelerator -> GPU."
    )

GPU_COUNT = torch.cuda.device_count()
print("CUDA GPUs detected:", GPU_COUNT)

for i in range(GPU_COUNT):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name} | VRAM: {props.total_memory / 1024**3:.2f} GB")

subprocess.run(["nvidia-smi"], check=False)


## 3. Install Real-ESRGAN


In [ ]:
%pip install -q --upgrade pip
%pip install -q basicsr facexlib gfpgan opencv-python-headless

import os
import shutil

REPO = "/kaggle/working/Real-ESRGAN"

if os.path.exists(REPO):
    shutil.rmtree(REPO)

!git clone --depth 1 https://github.com/xinntao/Real-ESRGAN.git /kaggle/working/Real-ESRGAN

%pip install -q -r /kaggle/working/Real-ESRGAN/requirements.txt
%pip install -q -e /kaggle/working/Real-ESRGAN

print("Real-ESRGAN installed.")


## 4. Fix BasicSR / modern torchvision compatibility

Kaggle's newer torchvision versions no longer provide the old
`torchvision.transforms.functional_tensor` import used by some BasicSR releases.
This patch changes it to the current `torchvision.transforms.functional` location.


In [ ]:
import os

degradations_file = "/usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py"

if os.path.exists(degradations_file):
    with open(degradations_file, "r") as f:
        text = f.read()

    old = "from torchvision.transforms.functional_tensor import rgb_to_grayscale"
    new = "from torchvision.transforms.functional import rgb_to_grayscale"

    if old in text:
        text = text.replace(old, new)
        with open(degradations_file, "w") as f:
            f.write(text)
        print("BasicSR compatibility patch applied.")
    elif new in text:
        print("BasicSR compatibility patch already present.")
    else:
        print("Expected import was not found; continuing.")
else:
    print("BasicSR degradations.py was not found at the expected path.")


## 5. Download the x2 model


In [ ]:
import os
import urllib.request

WEIGHTS_DIR = os.path.join(REPO, "weights")
os.makedirs(WEIGHTS_DIR, exist_ok=True)

WEIGHTS_PATH = os.path.join(WEIGHTS_DIR, "RealESRGAN_x2plus.pth")
WEIGHTS_URL = "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth"

if not os.path.exists(WEIGHTS_PATH):
    print("Downloading RealESRGAN_x2plus.pth...")
    urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS_PATH)

print("Model:", WEIGHTS_PATH)
print("Size:", round(os.path.getsize(WEIGHTS_PATH) / 1024**2, 1), "MB")


## 6. Inspect the input video


In [ ]:
import json
import subprocess
import os

if not os.path.isfile(INPUT_VIDEO):
    raise FileNotFoundError(f"Video not found: {INPUT_VIDEO}")

probe = subprocess.run(
    [
        "ffprobe", "-v", "error",
        "-select_streams", "v:0",
        "-show_entries",
        "stream=width,height,r_frame_rate,avg_frame_rate,nb_frames,duration",
        "-of", "json",
        INPUT_VIDEO
    ],
    capture_output=True, text=True, check=True
)

info = json.loads(probe.stdout)["streams"][0]

print("Resolution:", info.get("width"), "x", info.get("height"))
print("FPS:", info.get("avg_frame_rate") or info.get("r_frame_rate"))
print("Duration:", info.get("duration"), "seconds")
print("Frames:", info.get("nb_frames", "unknown"))


## 7. Extract frames

This creates temporary lossless PNG frames. They are deleted by the optional cleanup cell after the final video is checked.


In [ ]:
import os
import subprocess
import shutil

FRAMES_DIR = os.path.join(WORK_DIR, "frames")
UPSCALED_DIR = os.path.join(WORK_DIR, "upscaled")
GPU0_INPUT = os.path.join(WORK_DIR, "gpu0_input")
GPU1_INPUT = os.path.join(WORK_DIR, "gpu1_input")
GPU0_OUTPUT = os.path.join(WORK_DIR, "gpu0_output")
GPU1_OUTPUT = os.path.join(WORK_DIR, "gpu1_output")

# Start clean so rerunning this cell does not mix old frames with new ones.
if os.path.isdir(WORK_DIR):
    shutil.rmtree(WORK_DIR)

for d in [FRAMES_DIR, UPSCALED_DIR, GPU0_INPUT, GPU1_INPUT, GPU0_OUTPUT, GPU1_OUTPUT]:
    os.makedirs(d, exist_ok=True)

subprocess.run(
    [
        "ffmpeg", "-y",
        "-i", INPUT_VIDEO,
        "-vsync", "0",
        os.path.join(FRAMES_DIR, "frame%08d.png")
    ],
    check=True
)

frame_files = sorted(
    f for f in os.listdir(FRAMES_DIR)
    if f.lower().endswith(".png")
)

print(f"Extracted {len(frame_files)} frames.")


## 8. Split frames between the available T4 GPUs

If Kaggle gives this session two GPUs, the frames are split approximately 50/50.
If only one GPU is available, everything runs on GPU 0.


In [ ]:
import os
import shutil

# Clean split/output directories in case this cell is rerun.
for d in [GPU0_INPUT, GPU1_INPUT, GPU0_OUTPUT, GPU1_OUTPUT]:
    if os.path.isdir(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

use_two_gpus = torch.cuda.device_count() >= 2

mid = (len(frame_files) + 1) // 2 if use_two_gpus else len(frame_files)

groups = [
    (frame_files[:mid], GPU0_INPUT),
    (frame_files[mid:], GPU1_INPUT),
]

for files_for_gpu, destination in groups:
    for filename in files_for_gpu:
        src = os.path.join(FRAMES_DIR, filename)
        dst = os.path.join(destination, filename)
        os.symlink(src, dst)

print("Using two GPUs:", use_two_gpus)
print("GPU 0 frames:", len(groups[0][0]))
print("GPU 1 frames:", len(groups[1][0]) if use_two_gpus else 0)


## 9. AI upscale


In [ ]:
import os
import subprocess
import time

def upscale_process(input_dir, output_dir, visible_gpu, label):
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(visible_gpu)

    cmd = [
        "python",
        f"{REPO}/inference_realesrgan.py",
        "-n", MODEL,
        "-i", input_dir,
        "-o", output_dir,
        "-s", str(SCALE),
        "--suffix", "out",
        "--tile", "512"
    ]

    print(f"Starting {label} on physical GPU {visible_gpu}...")
    return subprocess.Popen(cmd, env=env)

start = time.time()

p0 = upscale_process(GPU0_INPUT, GPU0_OUTPUT, 0, "GPU 0")

if use_two_gpus:
    p1 = upscale_process(GPU1_INPUT, GPU1_OUTPUT, 1, "GPU 1")
    rc1 = p1.wait()
else:
    p1 = None
    rc1 = 0

rc0 = p0.wait()

if rc0 != 0:
    raise RuntimeError(f"GPU 0 Real-ESRGAN process failed with exit code {rc0}")

if rc1 != 0:
    raise RuntimeError(f"GPU 1 Real-ESRGAN process failed with exit code {rc1}")

elapsed = time.time() - start

print(f"AI upscale finished in {elapsed / 60:.1f} minutes.")


## 10. Combine the upscaled frames


In [ ]:
import os
import shutil

# Combine both GPU output folders into one output folder.
if os.path.isdir(UPSCALED_DIR):
    shutil.rmtree(UPSCALED_DIR)
os.makedirs(UPSCALED_DIR, exist_ok=True)

all_outputs = []

for source_dir in [GPU0_OUTPUT, GPU1_OUTPUT]:
    for filename in os.listdir(source_dir):
        if filename.lower().endswith(".png"):
            src = os.path.join(source_dir, filename)
            dst = os.path.join(UPSCALED_DIR, filename)
            shutil.copy2(src, dst)
            all_outputs.append(filename)

all_outputs.sort()

expected = len(frame_files)
actual = len(all_outputs)

print("Expected frames:", expected)
print("Upscaled frames:", actual)

if actual != expected:
    raise RuntimeError("Frame count mismatch. Do not rebuild the video.")

print("All frames are ready.")


## 11. Rebuild the 4K MP4 and preserve audio


In [ ]:
import subprocess
import os

fps_result = subprocess.run(
    [
        "ffprobe", "-v", "error",
        "-select_streams", "v:0",
        "-show_entries", "stream=avg_frame_rate",
        "-of", "default=noprint_wrappers=1:nokey=1",
        INPUT_VIDEO
    ],
    capture_output=True, text=True, check=True
)

fps = fps_result.stdout.strip()

frame_pattern = os.path.join(UPSCALED_DIR, "frame%08d_out.png")

cmd = [
    "ffmpeg", "-y",
    "-framerate", fps,
    "-i", frame_pattern,
    "-i", INPUT_VIDEO,
    "-map", "0:v:0",
    "-map", "1:a?",
    "-c:v", "libx264",
    "-crf", str(CRF),
    "-preset", PRESET,
    "-pix_fmt", "yuv420p",
    "-c:a", "copy",
    "-shortest",
    OUTPUT_VIDEO
]

print("Rebuilding:", OUTPUT_VIDEO)
subprocess.run(cmd, check=True)

print("Created:", OUTPUT_VIDEO)
print("Size (MB):", round(os.path.getsize(OUTPUT_VIDEO) / 1024**2, 2))


## 12. Verify the finished video


In [ ]:
import subprocess
import json

probe = subprocess.run(
    [
        "ffprobe", "-v", "error",
        "-show_streams",
        "-show_format",
        "-of", "json",
        OUTPUT_VIDEO
    ],
    capture_output=True, text=True, check=True
)

data = json.loads(probe.stdout)

for stream in data.get("streams", []):
    if stream.get("codec_type") == "video":
        print("Video:", stream.get("width"), "x", stream.get("height"))
        print("Video codec:", stream.get("codec_name"))
        print("FPS:", stream.get("avg_frame_rate"))
    elif stream.get("codec_type") == "audio":
        print("Audio:", stream.get("codec_name"), stream.get("sample_rate"), "Hz")

print("Output size (MB):", round(os.path.getsize(OUTPUT_VIDEO) / 1024**2, 2))


## 13. Download


In [ ]:
from IPython.display import FileLink, display

display(FileLink(
    OUTPUT_VIDEO,
    result_html_prefix="Download upscaled 4K video: "
))


## Optional cleanup

Run this **only after downloading and checking the output**. It deletes the temporary extracted/upscaled PNGs and frees Kaggle working storage.


In [ ]:
import shutil

if os.path.isdir(WORK_DIR):
    shutil.rmtree(WORK_DIR)
    print("Temporary frames and intermediate files deleted.")

if os.path.exists(OUTPUT_VIDEO):
    print("Final video remains:", OUTPUT_VIDEO)
